# Sovereign Workbench — Agent-Protocol LoRA

**SIH26117 · Sovereign On-Premise Agentic AI Workbench (MRPL)**

Fine-tunes a **Qwen3** model so it reliably speaks our agent protocol: one JSON
object per turn, the right tool, correctly formed arguments. Trains either tier
the workbench actually runs (`config.yaml` → `models:`) — set `MODEL_SIZE` in
§4b, which is the only place the tier is chosen:

- `qwen3:4b`  — the `general` model and the router's own tie-break call
- `qwen3:14b` — the `document`/`code` models. This is the one making the
  protocol-formatting decision on the highest-stakes tasks, so it is the
  higher-value fine-tune if you can only do one.

## Why this and not something else

The workbench already supplies the knowledge and the arithmetic — RAG retrieves
the SOP clause, the sandbox runs the calculation. What is left for the model is
narrow and mechanical: **choose a tool and format the call**. That is the part a
small model gets wrong, it is the single biggest live-demo risk, and it is
exactly what a LoRA fixes cheaply.

We are *not* training in refinery knowledge. Facts come from the corpus at
inference time, with citations. A model that memorised thresholds would be worse
— it could state one without a source.

## The sovereignty rule this notebook obeys

Training on a rented GPU is fine; the proposal permits HF GPUs for development.
What must never happen is the **runtime** calling out. So this notebook ends by
producing **GGUF weights + an Ollama Modelfile** that run entirely on the demo
machine. Nothing here becomes a runtime dependency.

## What you need

| Tier | VRAM (QLoRA) | Disk (whole pipeline) | System RAM (at merge) |
|---|---|---|---|
| `4b`  | 16 GB | ~26 GB | ~8 GB |
| `14b` | 24 GB, tight; 40 GB comfortable | **~92 GB** | **~28 GB** |

The disk and RAM columns are the ones that catch people out. A 14B run needs
roughly 92 GB of disk across the pipeline and 28 GB of system RAM at the merge
step — a box with ample VRAM but a 50 GB disk trains for forty minutes and then
dies after the GPU time is already spent. §4b refuses to start in that case.

- `data/training/agent_sft.jsonl`, produced by `scripts/build_training_data.py`.

## Order of cells

1. Environment check → 2. Install → 3. Get the dataset → 4. Inspect it →
4b. **Choose the tier + resource gate** → 5. Load the model → 6. LoRA config →
7. **Baseline eval** → 8. Train → 9. **Post-train eval** → 10. Merge →
11. GGUF + Modelfile → 12. Ship it back

To train both tiers, run the notebook twice end to end with `MODEL_SIZE` set to
`"4b"` then `"14b"` - every output path is derived from the tier, so the two
runs never collide or overwrite each other.


## 1 · Environment check

Stop early if there is no GPU, rather than 40 minutes in.

In [ ]:
import os

# Must be set before torch initialises its CUDA allocator, i.e. before the
# import in ss2. Reduces fragmentation, which is what turns "enough memory in
# total" into an OOM on a tight card. PyTorch's own OOM message suggests it.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import subprocess, sys, platform

print("python :", sys.version.split()[0])
print("system :", platform.platform())

try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as exc:
    print(f"nvidia-smi unavailable: {exc}")
    print("\nThis notebook needs a GPU. On HF, pick a GPU Space or a GPU-backed notebook.")


## 2 · Install

Pinned deliberately. `trl` and `peft` move fast and their `SFTConfig` argument
names change between minor versions; an unpinned install is the most common way
this notebook breaks months later.

In [ ]:
%pip install -q --upgrade pip
%pip install -q "torch>=2.4" "transformers>=4.57" "trl>=0.23" "peft>=0.17" "datasets>=3.0" "accelerate>=1.0" "bitsandbytes>=0.44" sentencepiece protobuf psutil

import torch, transformers, trl, peft
print("torch       ", torch.__version__, "| cuda:", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("trl         ", trl.__version__)
print("peft        ", peft.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    # GiB, because that is the unit CUDA OOM messages use - mixing GB and
    # GiB across cells is how a 16 GB card ends up looking like 15.8.
    print(f"gpu          {props.name}  {props.total_memory/2**30:.2f} GiB"
          f"  | bf16: {torch.cuda.is_bf16_supported()}")

## 3 · Get the dataset

Three ways, in order of preference:

1. **Clone the repo** and regenerate — guarantees the training prompt matches
   the prompt `src/core/prompts.py` sends at inference. If those two drift, the
   fine-tune teaches a format the runtime never uses and buys you nothing.
2. **Upload** `data/training/agent_sft.jsonl` next to this notebook.
3. Pull it from a HF dataset repo you pushed earlier.

In [ ]:
import os, json
from pathlib import Path

REPO_URL  = os.environ.get("SOVEREIGN_REPO", "https://github.com/hs-zz27/sih.git")
DATA_PATH = Path("agent_sft.jsonl")

if not DATA_PATH.exists():
    if Path("sih").exists() or os.system(f"git clone --depth 1 {REPO_URL} sih") == 0:
        # Regenerate from the live prompt + tool registry.
        rc = os.system("cd sih && pip install -q pyyaml pydantic numpy && "
                       "python scripts/build_training_data.py --n 800")
        candidate = Path("sih/data/training/agent_sft.jsonl")
        if rc == 0 and candidate.exists():
            DATA_PATH.write_text(candidate.read_text(encoding="utf-8"), encoding="utf-8")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "No agent_sft.jsonl. Upload it, or set SOVEREIGN_REPO to a reachable clone URL."
    )

rows = [json.loads(line) for line in DATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"{len(rows)} examples loaded from {DATA_PATH}")

## 4 · Inspect before you train

Read one example end to end. Most bad fine-tunes are visible here.

In [ ]:
example = rows[0]
for message in example["messages"]:
    print("=" * 78)
    print(message["role"].upper())
    print("=" * 78)
    print(message["content"][:1200])
    print()

# Sanity: every assistant turn must be a single valid JSON object, because that
# is precisely what we are teaching.
bad = 0
for row in rows:
    for message in row["messages"]:
        if message["role"] == "assistant":
            try:
                json.loads(message["content"])
            except json.JSONDecodeError:
                bad += 1
print(f"malformed assistant turns: {bad}  (must be 0)")
assert bad == 0

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(rows).train_test_split(test_size=0.05, seed=42)
print(dataset)

## 4b · Choose the tier, and check the box can actually finish

**This is the only cell where you choose which model to train.** Everything
downstream - batch size, learning rate, output directory names, GGUF filenames,
the Modelfile - is derived from `MODEL_SIZE`.

It also refuses to continue if this machine cannot finish the job: VRAM, disk,
and system RAM, checked before anything downloads.

**Units are GiB here, not GB**, because that is what CUDA's own OOM messages
use. A "16 GB" card reports 14.74 GiB (15.83 decimal GB) - so a threshold
written as `16` in decimal GB can never be met by a 16 GB card. That mistake
is why an earlier version of this gate rejected hardware that was actually fine.

| | VRAM needed | disk (whole pipeline) | system RAM (at merge) |
|---|---|---|---|
| `4b`  | ~8 GiB  | ~26 GB | ~8 GB |
| `14b` | ~18 GiB | ~92 GB | ~28 GB |

It distinguishes two different VRAM failures, because they have opposite fixes:

- **not enough total** - the card is too small for this tier. Use a bigger GPU,
  or drop to `4b`.
- **enough total, little free** - something still holds the memory, almost
  always a failed earlier attempt in the same kernel. Restarting the kernel
  fixes it; a bigger GPU does not.


In [ ]:
import shutil

import torch

GIB = 2 ** 30

# --- The one switch that selects the tier -----------------------------------
# Matches config.yaml's models: block - "4b" is what `general` runs, "14b" is
# what `document` and `code` run.
MODEL_SIZE = "4b"           # <- "4b" or "14b". Nothing else to change.

# min_vram_gib is measured, deliberately, in GiB - the unit CUDA reports. It is
# the working set actually needed (4-bit weights + quantisation state + the
# fp32 casts prepare_model_for_kbit_training does + activations at the batch
# size and sequence length below), not the marketing size of a card.
PRESETS = {
    "4b": dict(
        base_model="Qwen/Qwen3-4B",
        params_b=4,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,     # effective batch 16
        learning_rate=2e-4,
        lora_r=16,
        lora_alpha=32,
        min_vram_gib=8,                    # ~2 GiB weights + activations + headroom
        comfortable_gpu="a 16 GB card (T4, L4) is fine",
    ),
    "14b": dict(
        base_model="Qwen/Qwen3-14B",
        params_b=14,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,    # effective batch 16, smaller per-step footprint
        learning_rate=1e-4,                # lower LR is the usual call for a larger base
        lora_r=16,
        lora_alpha=32,
        min_vram_gib=18,                   # ~6.5 GiB weights, and the fp32 cast alone wants ~3
        comfortable_gpu="a 24 GB card (L4, A10G) minimum; 48 GB (L40S) comfortable",
    ),
}

if MODEL_SIZE not in PRESETS:
    raise ValueError(f"MODEL_SIZE must be one of {list(PRESETS)}, got {MODEL_SIZE!r}")

cfg = PRESETS[MODEL_SIZE]
BASE_MODEL = cfg["base_model"]
MODEL_SLUG = BASE_MODEL.split("/")[-1].lower().replace(".", "")   # e.g. "qwen3-4b"
PARAMS_B = cfg["params_b"]
OTHER_TIER = "14b" if MODEL_SIZE == "4b" else "4b"

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU visible to torch. Re-check ss1's nvidia-smi output - this "
        "notebook cannot train on CPU in any reasonable time."
    )

# --- dtype: Turing (T4, sm_75) has no bf16. Ampere and later do. -------------
# Picking this wrong does not fail fast; it either errors deep inside the
# trainer or silently runs an emulated path at a fraction of the speed.
capability = torch.cuda.get_device_capability(0)
USE_BF16 = torch.cuda.is_bf16_supported() and capability[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

# --- VRAM -------------------------------------------------------------------
torch.cuda.empty_cache()          # release anything an earlier attempt left cached
vram_free_gib, vram_total_gib = (x / GIB for x in torch.cuda.mem_get_info())
need_gib = cfg["min_vram_gib"]

# --- disk / RAM -------------------------------------------------------------
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
except Exception:
    ram_gb = None

# hf cache (the bf16 base is downloaded even for a 4-bit load) + merged
# safetensors + f16 GGUF + Q4 GGUF. The f16 is deleted in ss11, but it has to
# exist first, so the peak is what matters.
disk_need = PARAMS_B * 2 * 3 + PARAMS_B * 0.6
ram_need = PARAMS_B * 2        # merge loads the base on CPU
disk_free = shutil.disk_usage(".").free / 1e9

print(f"tier            : {MODEL_SIZE}  ({BASE_MODEL})")
print(f"gpu             : {torch.cuda.get_device_name(0)}  (sm_{capability[0]}{capability[1]})")
print(f"dtype           : {'bfloat16' if USE_BF16 else 'float16'}"
      f"{'' if USE_BF16 else '  <- this GPU has no bf16; fp16 selected automatically'}")
print(f"VRAM needed     : ~{need_gib} GiB")
print(f"VRAM free/total : {vram_free_gib:.2f} / {vram_total_gib:.2f} GiB")
print(f"disk required   : ~{disk_need:.0f} GB peak")
print(f"disk available  : {disk_free:.0f} GB")
print(f"RAM required    : ~{ram_need} GB (at the merge step, ss10)")
print(f"RAM available   : {f'{ram_gb:.0f} GB' if ram_gb else 'unknown (psutil not installed)'}")

problems = []

# Order matters. "Enough card, but occupied" and "card too small" have opposite
# fixes, and reporting the wrong one sends people to buy hardware they already
# have.
if vram_total_gib < need_gib:
    problems.append(
        f"VRAM (card too small): the {MODEL_SIZE} tier needs ~{need_gib} GiB, this "
        f"GPU has {vram_total_gib:.2f} GiB in total. Use a bigger GPU - "
        f"{cfg['comfortable_gpu']}"
        + (f" - or set MODEL_SIZE = \"{OTHER_TIER}\" above." if MODEL_SIZE == "14b"
           else ". The 4b tier is already the smaller of the two.")
    )
elif vram_free_gib < need_gib:
    held_gib = vram_total_gib - vram_free_gib
    problems.append(
        f"VRAM (card is big enough, but {held_gib:.2f} GiB of it is already held): "
        f"{vram_free_gib:.2f} GiB free, need ~{need_gib} GiB. This is almost always a "
        f"failed earlier run still holding memory in this kernel - empty_cache() "
        f"did not recover it, so restart the kernel/runtime and re-run from ss1. "
        f"A bigger GPU would NOT fix this."
    )

if disk_free < disk_need:
    problems.append(
        f"Disk: need ~{disk_need:.0f} GB, have {disk_free:.0f} GB. Free space now - "
        f"otherwise the run dies at the merge or GGUF step, after training has "
        f"finished and the GPU time is spent."
    )
if ram_gb is not None and ram_gb < ram_need:
    problems.append(
        f"RAM: the merge step loads the base on CPU and needs ~{ram_need} GB, have "
        f"{ram_gb:.0f} GB. It will be OOM-killed. Use a box with more RAM, or merge "
        f"on the GPU (device_map={{'': 0}} in ss10) if VRAM allows."
    )

if problems:
    print()
    for problem in problems:
        print("BLOCKER:", problem)
    raise RuntimeError("Resource preflight failed - see above. Fix before training, not after.")

print("\nResource preflight OK.")


## 5 · Load the selected model in 4-bit

Uses the tier and dtype resolved in ss4b - nothing to set here.

QLoRA: frozen 4-bit base, small trainable adapters. The compute dtype follows
the GPU: bf16 on Ampere and later, fp16 on Turing (a T4 has no bf16 at all, and
hardcoding it there fails deep inside the trainer rather than here).


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# VRAM/disk/RAM and dtype were resolved in ss4b, before any download.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # correct for training; flip to left for batched generation

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    dtype=COMPUTE_DTYPE,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)
model.config.use_cache = False          # incompatible with gradient checkpointing
print(model.config.model_type, f"{model.num_parameters()/1e9:.2f}B params  (tier: {MODEL_SIZE})")
print(f"VRAM in use after load: {torch.cuda.memory_allocated()/2**30:.2f} GiB")


In [ ]:
# Confirm the chat template renders our conversation as expected. If this looks
# wrong, everything downstream is wrong.
rendered = tokenizer.apply_chat_template(
    dataset["train"][0]["messages"], tokenize=False, add_generation_prompt=False
)
print(rendered[:1500])

lengths = sorted(
    len(tokenizer.apply_chat_template(r["messages"], tokenize=True))
    for r in dataset["train"].select(range(min(200, len(dataset["train"]))))
)
longest = lengths[-1]
print(f"\ntokens per example: mean {sum(lengths)//len(lengths)}, max {longest}")

# Size the sequence window to the data rather than pinning 4096. Activation
# memory scales with this, and on a 16 GB card the difference between 4096 and
# what the corpus actually needs is the difference between fitting and not.
# Round up to a power of two, with headroom, and never exceed 4096.
MAX_SEQ_LEN = 512
while MAX_SEQ_LEN < min(4096, int(longest * 1.25)):
    MAX_SEQ_LEN *= 2

print(f"MAX_SEQ_LEN       : {MAX_SEQ_LEN}  (longest example {longest} tokens)")
print("truncated         :", sum(1 for n in lengths if n > MAX_SEQ_LEN))


## 6 · LoRA configuration

`r` and `alpha` come from the tier preset chosen in ss5 (both tiers default to
`r=16`, ample for a formatting-and-selection task - we are shaping output
structure, not installing new knowledge). Targeting attention **and** MLP
projections gives the adapter enough capacity to change tool-choice behaviour,
not just phrasing, regardless of tier.


In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Gradient checkpointing + a frozen 4-bit base means the inputs to the first
# checkpointed block carry no grad, and the first backward pass dies with
# "element 0 of tensors does not require grad". prepare_model_for_kbit_training
# usually sets this, but it is version-dependent and the failure happens at
# step 1 of training - so make it explicit rather than hope.
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

peft_config = LoraConfig(
    r=cfg["lora_r"],
    lora_alpha=cfg["lora_alpha"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
print(peft_config)


## 7 · Baseline eval — measure before you change anything

This is the cell that makes the exercise honest. We ask the **untrained** model
to produce agent turns and score them with the workbench's own parser:

- **parse rate** — did we get one valid decision object?
- **valid tool rate** — is the named tool one that actually exists?

If the baseline is already ~100%, the LoRA is unnecessary and you should say so
rather than train anyway. Record these two numbers; they are the honest before/
after for the pitch.

In [ ]:
import re, json

# The workbench's own parsing rules, inlined so this cell works without the repo.
_THINK = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
_OPEN  = re.compile(r"<think>.*$",        re.DOTALL | re.IGNORECASE)
_FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL)

TOOLS = {"read_file", "write_file", "list_files", "search_documents", "run_python"}


def parse_decision(text: str):
    """Mirror of src/core/agent.py::_parse_decision. Returns dict or None."""
    if not text or not text.strip():
        return None
    text = _OPEN.sub("", _THINK.sub("", text)).strip()

    candidates = []
    if text.startswith("{"):
        candidates.append(text)
    candidates += [m.group(1).strip() for m in _FENCE.finditer(text)]

    depth, start, in_str, esc = 0, -1, False, False
    for i, ch in enumerate(text):
        if in_str:
            if esc: esc = False
            elif ch == "\\": esc = True
            elif ch == '"': in_str = False
            continue
        if ch == '"': in_str = True
        elif ch == "{":
            if depth == 0: start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start >= 0:
                candidates.append(text[start:i + 1]); start = -1
            elif depth < 0: depth = 0

    for candidate in candidates:
        try:
            payload = json.loads(candidate)
        except json.JSONDecodeError:
            continue
        if isinstance(payload, dict) and ("tool" in payload or "final_answer" in payload):
            return payload
    return None


@torch.no_grad()
def generate_turns(model, tokenizer, rows, limit=40, max_new_tokens=192):
    """Generate the model's next turn for each held-out example.

    Re-enables the KV cache for the duration. Training needs use_cache=False
    (it is incompatible with gradient checkpointing), but generating with the
    cache off is quadratic in sequence length - on a 14B that is the difference
    between a minute and something that looks like a hung notebook.
    """
    model.eval()
    previous_cache_setting = model.config.use_cache
    model.config.use_cache = True
    tokenizer.padding_side = "left"
    outputs = []
    for row in rows[:limit]:
        messages = row["messages"]
        # Cut at the first assistant turn: ask the model to produce it.
        cut = next(i for i, m in enumerate(messages) if m["role"] == "assistant")
        prompt = tokenizer.apply_chat_template(
            messages[:cut], tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        generated = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False,                      # greedy: matches demo settings
            pad_token_id=tokenizer.pad_token_id,
        )
        text = tokenizer.decode(generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        outputs.append((text, json.loads(messages[cut]["content"])))
    tokenizer.padding_side = "right"
    model.config.use_cache = previous_cache_setting
    return outputs


def score(outputs):
    parsed_ok = valid_tool = tool_match = 0
    for text, expected in outputs:
        decision = parse_decision(text)
        if decision is None:
            continue
        parsed_ok += 1
        tool = decision.get("tool")
        if tool is None or tool in TOOLS:
            valid_tool += 1
        if tool == expected.get("tool"):
            tool_match += 1
    n = len(outputs)
    return {
        "n": n,
        "parse_rate": round(parsed_ok / n, 3),
        "valid_tool_rate": round(valid_tool / n, 3),
        "tool_match_rate": round(tool_match / n, 3),
    }


held_out = list(dataset["test"])
baseline_outputs = generate_turns(model, tokenizer, held_out)
BASELINE = score(baseline_outputs)
print("BASELINE:", BASELINE)
print("\n--- sample generation ---\n", baseline_outputs[0][0][:600])

## 8 · Train

`assistant_only_loss=True` means loss is computed on the model's turns alone.
Without it the model also learns to predict our tool observations - wasted
capacity on text it will never have to produce.

Batch size and gradient accumulation come from the ss5 preset: the 14B tier
trains at a smaller per-step footprint (batch 1, more accumulation steps) to
fit in less VRAM, at the same effective batch size as the 4B tier.


In [ ]:
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = f"{MODEL_SLUG}-sovereign-agent-lora"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=cfg["per_device_train_batch_size"],
    gradient_accumulation_steps=cfg["gradient_accumulation_steps"],
    learning_rate=cfg["learning_rate"],
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    bf16=USE_BF16,          # ss4b: Turing has no bf16, so fp16 there instead
    fp16=not USE_BF16,
    max_length=MAX_SEQ_LEN,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",                       # no telemetry, on principle
    seed=42,
)

# assistant_only_loss landed in trl 0.20; degrade gracefully on older versions.
try:
    sft_config.assistant_only_loss = True
except Exception as exc:
    print("assistant_only_loss unavailable, training on full sequences:", exc)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.model.print_trainable_parameters()
print(f"tier: {MODEL_SIZE}  |  effective batch size: "
      f"{cfg['per_device_train_batch_size'] * cfg['gradient_accumulation_steps']}"
      f"  |  dtype: {'bf16' if USE_BF16 else 'fp16'}  |  max_len: {MAX_SEQ_LEN}")


In [ ]:
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\ntrain loss:", round(train_result.training_loss, 4))
print("adapter saved to:", OUTPUT_DIR)

## 9 · Post-train eval

Same held-out set, same scorer. This is the number that justifies the work.

In [ ]:
trainer.model.config.use_cache = True
trained_outputs = generate_turns(trainer.model, tokenizer, held_out)
TRAINED = score(trained_outputs)

print(f"{'metric':<20} {'baseline':>10} {'trained':>10} {'delta':>10}")
print("-" * 52)
for key in ("parse_rate", "valid_tool_rate", "tool_match_rate"):
    before, after = BASELINE[key], TRAINED[key]
    print(f"{key:<20} {before:>10.3f} {after:>10.3f} {after - before:>+10.3f}")

print("\n--- sample generation ---\n", trained_outputs[0][0][:600])

if TRAINED["parse_rate"] <= BASELINE["parse_rate"]:
    print("\nNo improvement in parse rate. Say so rather than shipping the adapter:")
    print("the base model may already be reliable enough, in which case skip the LoRA.")

## 10 · Merge the adapter

The demo runs one set of weights in Ollama, so the adapter is merged into the
base model rather than loaded separately.

In [ ]:
import gc, os, shutil, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

MERGED_DIR = f"{MODEL_SLUG}-sovereign-agent-merged"

# --- Free the training model first, or the merge competes with it for memory.
# Guarded so re-running this cell after a failure does not die on NameError.
for _name in ("trainer", "model", "base", "merged"):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --- Fail fast if there is not enough room, rather than 20 minutes into a
# 28 GB write. This is the step that kills 14B runs on boxes that have plenty
# of VRAM but a modest disk.
_need_gb = PARAMS_B * 2                      # merged safetensors, bf16
_free_gb = shutil.disk_usage(".").free / 1e9
print(f"merge needs ~{_need_gb} GB disk, {_free_gb:.0f} GB free")
if _free_gb < _need_gb * 1.15:
    raise RuntimeError(
        f"Not enough disk to write the merged model: need ~{_need_gb} GB, "
        f"have {_free_gb:.0f} GB. Free space (the HF cache under "
        f"~/.cache/huggingface is usually the biggest thing here) and re-run "
        f"this cell - the adapter in {OUTPUT_DIR} is already saved, so nothing "
        f"is lost."
    )

# low_cpu_mem_usage keeps peak RAM near the model size instead of roughly
# double it. A 14B base at 2 bytes/param is ~28 GB of system RAM either way, so if this
# cell gets OOM-killed, that is why - use a box with more RAM, or merge on the
# GPU by setting device_map={"": 0} if VRAM allows.
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=COMPUTE_DTYPE,        # same dtype the adapter was trained in
    device_map="cpu",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

del base, merged
gc.collect()

print("merged weights ->", MERGED_DIR)
print(f"disk free now: {shutil.disk_usage('.').free / 1e9:.0f} GB")


## 11 · GGUF + Ollama Modelfile - the part that keeps the claim true

Converts to GGUF and writes the Modelfile the demo machine uses. After this the
workbench needs **nothing** from HuggingFace at runtime.

`Q4_K_M` is the right quantisation for either tier on a laptop or a single-GPU
demo box:

| Tier | Approx. Q4_K_M size | Matches |
|---|---|---|
| `4b`  | ~2.5 GB | `config.yaml` → `models.general` |
| `14b` | ~8-9 GB | `config.yaml` → `models.document` / `models.code` |


In [ ]:
import shutil, subprocess, sys
from pathlib import Path


def run(command: str, what: str) -> None:
    """Run a shell step and stop the notebook if it fails.

    The original used os.system() and ignored the return code, so a failed
    conversion carried on and wrote a Modelfile pointing at a GGUF that did not
    exist - the error only surfaced later on the demo machine.
    """
    print(f"$ {command}")
    if subprocess.call(command, shell=True) != 0:
        raise RuntimeError(f"{what} failed. Fix this before continuing.")


GGUF_F16 = f"sovereign-agent-{MODEL_SLUG}-f16.gguf"
GGUF_Q4  = f"sovereign-agent-{MODEL_SLUG}-q4_k_m.gguf"

# Disk check: f16 is ~2 bytes/param, Q4_K_M ~0.6. Both exist at once until the
# f16 is deleted below, so the peak is what matters.
_need_gb = PARAMS_B * 2 + PARAMS_B * 0.6
_free_gb = shutil.disk_usage(".").free / 1e9
print(f"GGUF conversion needs ~{_need_gb:.0f} GB peak, {_free_gb:.0f} GB free")
if _free_gb < _need_gb * 1.15:
    raise RuntimeError(
        f"Not enough disk for GGUF conversion: need ~{_need_gb:.0f} GB peak, "
        f"have {_free_gb:.0f} GB. The merged model in {MERGED_DIR} is already "
        f"saved - free space and re-run this cell."
    )

if not Path("llama.cpp").exists():
    run("git clone --depth 1 https://github.com/ggerganov/llama.cpp", "llama.cpp clone")

# The requirements path moved between llama.cpp versions; try both.
for req in ("llama.cpp/requirements/requirements-convert_hf_to_gguf.txt",
            "llama.cpp/requirements.txt"):
    if Path(req).exists():
        subprocess.call(f"{sys.executable} -m pip install -q -r {req}", shell=True)
        break

run(f"{sys.executable} llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} "
    f"--outfile {GGUF_F16} --outtype f16", "GGUF f16 conversion")

# Quantise. Newer llama.cpp builds the tool via cmake; build it if absent.
_quantize = Path("llama.cpp/build/bin/llama-quantize")
if not _quantize.exists():
    run("cd llama.cpp && cmake -B build -DLLAMA_CURL=OFF && "
        "cmake --build build --config Release -j --target llama-quantize",
        "llama-quantize build")
run(f"./{_quantize} {GGUF_F16} {GGUF_Q4} Q4_K_M", "Q4_K_M quantisation")

if not Path(GGUF_Q4).exists():
    raise RuntimeError(f"{GGUF_Q4} was not produced despite a clean exit code.")

for path in (GGUF_F16, GGUF_Q4):
    if Path(path).exists():
        print(f"{path}: {Path(path).stat().st_size/1e9:.2f} GB")

# Reclaim the f16 intermediate - it is ~28 GB for a 14B and nothing downstream
# needs it. Keep it only if you plan to produce other quantisations.
KEEP_F16 = False
if not KEEP_F16 and Path(GGUF_F16).exists():
    Path(GGUF_F16).unlink()
    print(f"removed {GGUF_F16}; disk free now: {shutil.disk_usage('.').free/1e9:.0f} GB")


In [ ]:
MODELFILE = f"""FROM ./{GGUF_Q4}

# Sovereign Workbench agent model - {BASE_MODEL} + agent-protocol LoRA (merged).
# Tier: {MODEL_SIZE}. Runs entirely on the demo machine. No external service
# is contacted.

PARAMETER temperature 0
PARAMETER top_p 0.8
PARAMETER top_k 20
PARAMETER num_ctx 8192
PARAMETER repeat_penalty 1.05

# The agent supplies its own system prompt per task type (document / code /
# general), so none is baked in here.
"""

Path(f"Modelfile-{MODEL_SLUG}").write_text(MODELFILE, encoding="utf-8")
print(MODELFILE)


## 12 · Ship it back to the demo machine

Download `sovereign-agent-{MODEL_SLUG}-q4_k_m.gguf` and `Modelfile-{MODEL_SLUG}`
(both filenames include the tier you just trained), put them in one folder,
then **on the demo laptop**:

```bash
ollama create sovereign-agent-14b -f Modelfile-qwen3-14b   # or -4b / Modelfile-qwen3-4b
ollama run sovereign-agent-14b "hello"
```

Then point the workbench at it in `config.yaml`, matching the tier you trained
against the slot it actually fills:

```yaml
models:
  document: "sovereign-agent-14b"   # was qwen3:14b
  code:     "sovereign-agent-14b"   # was qwen3:14b
  general:  "sovereign-agent-4b"    # was qwen3:4b - only if you trained this tier too
```

If you only trained one tier, leave the other slot on its base model
(`qwen3:14b` / `qwen3:4b`) rather than pointing every slot at a model that was
never fine-tuned for that role.

Verify end to end with the network off:

```bash
python -m pytest tests -q
curl -s http://127.0.0.1:8000/api/health | python -m json.tool
python scripts/smoke_e2e.py          # compare parse rate against the base model - ss7/ss9 numbers
```

### Optional: push the adapter to the Hub

For versioning during development only. The demo never fetches it.


In [ ]:
PUSH = False   # set True to publish the adapter for versioning

if PUSH:
    from huggingface_hub import login, HfApi
    login()  # or set HF_TOKEN
    repo_id = "your-username/qwen35-4b-sovereign-agent-lora"
    HfApi().create_repo(repo_id, exist_ok=True)
    HfApi().upload_folder(folder_path=OUTPUT_DIR, repo_id=repo_id)
    print("pushed:", repo_id)
else:
    print("PUSH is False - nothing uploaded. The demo does not need the Hub.")

## Record the result

Write the before/after numbers into `HARDCODED.md` (or the pitch notes) with the
date, the dataset size and the commit that produced the data. If the LoRA did
not improve the parse rate, say that plainly and ship the base model — an honest
null result is defensible in Q&A, and an unexplained adapter is not.